<img src="./figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Beijing, China), Individual Contest](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Chicken_Counting/Chicken_Counting.ipynb)

# Chicken Counting

## **1. Problem Description**

As the leader of an AI research team collaborating with Silkie chicken farmers, you are tasked with solving a critical challenge in traditional free-range farming. Accurate counting of livestock is crucial for both farmers and insurance companies, as factors like disease outbreaks and predator invasions can significantly impact the survival rate of these chickens in a short time. While insurance coverage helps mitigate farming risks, the claims process requires precise counting of livestock losses. Your farmers have approached your team for help in developing more accurate, automated counting systems. The challenge before your research team is to develop an optimized Silkie chicken counting model using density estimation techniques that can provide reliable counts to support both farm management and insurance processes.

Your team has access to a pretrained feature extractor for Silkie chicken images, but you'll need to design and train the density estimation decoder to create a complete counting solution. Your task is to build upon this foundation by developing an effective decoder architecture and training strategy to achieve accurate chicken counts that farmers and insurance companies can rely on.

The below figure shows an image in the dataset, as well as the corresponding true density distribution and a predicted density distribution generated by the baseline model. The total density (sum of densities across all areas) is labeled.

<img src="./figs/Chicken Counting Fig 1.png" width="800">

## **2. Dataset**

The structure of the provided Silkie chicken image dataset is as follows:

```
datasets/
├── train/
│   └── A dataset with features:
│       ├── `image`: `PIL.Image` with RGB channels (3x720x1280)
│       └── `density`: a 2D array of shape 180x320
└── base.pth (Pretrained Model)


os.environ.get("DATA_PATH")/
├── test_a/
│   └── A dataset with features:
│       └── `image`: `PIL.Image` with RGB channels (3x720x1280)
└── test_b/
    └── A dataset with features:
        └── `image`: `PIL.Image` with RGB channels (3x720x1280)
```

(1) Training set location: `datasets`, files in this folder are used for model fine-tuning. It contains a train folder, which stores a dataset with 100 images and their corresponding density maps.

(2) Validation set (test_a) and Test set (test_b): These will be used to evaluate scores on Leaderboard A and Leaderboard B, respectively. They will be inaccessible to contestants. Only the score achieved on test set B will be used for final scoring.

(3) Datasets size:

- Training set: 100 images.
- Validation set: 100 images.
- Test set: 100 images.

(4) Validation set (test_a) and Test set (test_b) are not visible.

(5) Due to limits of computing resources, training density maps are reshaped to $1\times 180 \times 320$. **NOTE** the output density map can be viewed as a 2D real number matrix with shape $180\times 320$, the sum of all matrix values is the count of chickens.

## **3. Task**

Your task is to train your own model using the training data to predict density maps, thereby serving the purpose of chicken counting.

You may extend and optimize the given pretrained model to improve its count prediction accuracy. The pretrained model `base.pth` only contains the weights of the first four layers of the model (the feature extraction model), and the function `load_pretrained_weights_partial` in the baseline code can be used to load these partial weights into your model. You may construct a density decoder `DensityDecoder` and combine it with the pretrained feature extraction module to form a complete data prediction model.

```
class DensityDecoder(nn.Module):
    def __init__(self):
        #################################################
        # Your code here
        #################################################

    def forward(self, x):
        #################################################
        # Your code here
        #################################################
        return x
```

You can also build your own model without the pretrained model we provided.

This task is the continuation of Satellite Weather Forecasting. A kind remind is the UNET is easily to full GPU memory without any feature engineering. Then, the GPU memory error message will be reported.

Please follow these rules to achieve a score normally:

(1) Your model must output the predicted density map.

(2) Due to limits of computing resources, your output density map should be reshaped to $180 \times 320$. This is also the shape of target density maps provided in the train dataset.

## 4. Submission

Please submit a **submission.ipynb** that includes the following components:

（1）**Training Code**  

- Include the full training pipeline.

（2）**Evaluation Code**
- Evaluate your model on the validation set and test set.

- The output should be saved as **`submission.npz`**. This must be a valid `npz` file containing two arrays `pred_a` and `pred_b`, each with shape `100x1x180x320` (The evaluation script will also accept predictions in the shape of `100x180x320`, if you decide to squeeze the channel dimension).

  **Any result that does not meet the specified size will be considered invalid, resulting in an assessment score of zero.**
  
- Each element of the density map should be **no less than zero**, otherwise will result in an assessment score of zero.

## **5. Scoring**

You will be scored based on the mean relative error of your model. Relative error is defined by:

$$
\text{Relative Error} = \frac{|y_i - \hat{y}_i|}{|y_i|}
$$

where $y_i$ is the true total density for the $i$-th sample, and $\hat{y}_i$ is the predicted total density for the $i$-th sample.

Your final score before normalization will be calculated based on your mean relative error, as follows:

$$
\text{Score} = \exp(-\frac{1}{n} \sum_{i=1}^{n} \frac{|y_i - \hat{y}_i|}{y_i})
$$

## **6. Baseline & Training Set**

- Below you can find the baseline solution.
- The dataset is in `training_set` folder.
- The highest score by the Scientific Committee for this task is 0.89,  this score is used for score unification.
- The baseline score by the Scientific Committee for this task is 0.71, this score is used for score unification.

# ResUnet


In [8]:
import os
import cv2
import math
import json
import torch
import logging
import numpy as np
import pandas as pd
import torch.nn as nn
from tqdm.auto import tqdm
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms
import torch.optim

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
TARGET_SIZE = (180, 320)
INPUT_SIZE = (720, 1280)

In [35]:
class FeatureExtraction(nn.Module):
    def __init__(self):
        super(FeatureExtraction, self). __init__()
        resnet = models.resnet18(weights=None)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return x1, x2, x3, x4

class ChickenCounting(nn.Module):
    def __init__(self, base_weights_path=None):
        super(ChickenCounting, self).__init__()
        self.feature_extractor = FeatureExtraction()
        
        if base_weights_path and os.path.exists(base_weights_path):
            self.feature_extractor.load_state_dict(torch.load(base_weights_path, map_location='cpu'), strict=False)
            logging.info(f"Loaded base weights from {base_weights_path}")

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2) 
        self.conv1 = nn.Conv2d(256 + 256, 256, kernel_size=3, padding=1)
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(128 + 128, 128, kernel_size=3, padding=1)
        
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64 + 64, 64, kernel_size=3, padding=1)
        
        self.final_conv = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=1),
            nn.ReLU()
        )

    def forward(self, x):
        # 1. Encoder
        x1, x2, x3, x4 = self.feature_extractor(x)
        
        # 2. Decoder with automatic padding for shape mismatches
        
        # Level 1
        d1 = self.up1(x4)
        # Fix d1 to match x3
        if d1.shape[2:] != x3.shape[2:]:
            d1 = F.pad(d1, [0, x3.shape[3] - d1.shape[3], 0, x3.shape[2] - d1.shape[2]])
        d1 = torch.cat([d1, x3], dim=1) 
        d1 = F.relu(self.conv1(d1))
        
        # Level 2
        d2 = self.up2(d1)
        # Fix d2 to match x2
        if d2.shape[2:] != x2.shape[2:]:
            d2 = F.pad(d2, [0, x2.shape[3] - d2.shape[3], 0, x2.shape[2] - d2.shape[2]])
        d2 = torch.cat([d2, x2], dim=1)
        d2 = F.relu(self.conv2(d2))
        
        # Level 3
        d3 = self.up3(d2)
        # Fix d3 to match x1
        if d3.shape[2:] != x1.shape[2:]:
            d3 = F.pad(d3, [0, x1.shape[3] - d3.shape[3], 0, x1.shape[2] - d3.shape[2]])
        d3 = torch.cat([d3, x1], dim=1)
        d3 = F.relu(self.conv3(d3))
        
        # 3. Final Head
        out = self.final_conv(d3)
        
        # Ensure final output is exactly TARGET_SIZE (180, 320)
        if out.shape[2:] != (180, 320):
            out = F.interpolate(out, size=(180, 320), mode='bilinear', align_corners=False)
            
        return out

In [2]:
def train_model(model, train_loader, val_loader, num_epochs=20):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss(reduction='sum')
    model.to(DEVICE)
    
    best_mae = float('inf')
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            inputs, targets = batch['image'].to(DEVICE), batch['density'].to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_mae = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs, targets = batch['image'].to(DEVICE), batch['density'].to(DEVICE)
                outputs = model(inputs)
                
                pred_count = outputs.sum().item()
                true_count = targets.sum().item()
                val_mae += abs(pred_count - true_count)
        
        avg_mae = val_mae / len(val_loader.dataset)
        logging.info(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f}, Val MAE: {avg_mae:.4f}")
        
        if avg_mae < best_mae:
            best_mae = avg_mae
            torch.save(model.state_dict(), "best_model.pth")
            
    return model

def predict_set(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            inputs = batch['image'].to(DEVICE)
            outputs = model(inputs)
            all_preds.append(outputs.cpu().numpy())
    return np.concatenate(all_preds, axis=0)

In [5]:
from datasets import load_dataset

train_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Train2", 
                            data_dir="train",
                            split="train")  

image_transform = transforms.Compose([
    transforms.ToTensor(),
])

def collate_fn(batch, scale=100):
    return {
        "image": torch.stack([image_transform(item["image"]) for item in batch]),
        "density": torch.stack([torch.tensor(item["density"], dtype=DTYPE).unsqueeze(0) * scale for item in batch])
    }

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

model = ChickenCounting(base_weights_path="/kaggle/working/model.pth")

NameError: name 'transforms' is not defined

In [ ]:
import torch.optim as optim

################################################################################
# Experiment Settings
################################################################################
learning_rate = 1e-4
lr_decay = 1e-5
weight_decay = 0.0001
save_path = "model.pth"

epochs = 20

model = ChickenCounting().to(DEVICE)
print('load model success')

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=1 - lr_decay)

logging.info('Begin training single view model...')
train_model(model, train_loader, val_loader)
logging.info('Finished training single view model.')

In [11]:
def evaluate(model, val_loader, device, scale):
    model.eval()

    mse = 0.0
    mae = 0.0
    predict_num = 0.0
    true_num = 0.0
    rate = 0.0

    with torch.no_grad():  # Disable gradient calculation for inference
        for i, data in enumerate(val_loader, 0):
            inputs, targets = data["image"], data["density"]
            inputs = inputs.to(device).float()  # Move inputs to device and convert to float
            targets = targets.to(device).float()  # Move targets to device and convert to float

            # Get the model predictions
            outputs = model(inputs) / scale  # Adjusting for the scaling factor

            # Convert tensors to numpy for visualization and metrics calculation
            inputs_np = inputs.cpu().numpy()  # Convert inputs to numpy
            targets_np = targets.cpu().numpy()  # Convert targets to numpy
            outputs_np = outputs.cpu().numpy()  # Convert outputs to numpy
            # imshow_res(inputs_np, targets_np, outputs_np, scale)  # Uncomment to visualize results

            # Calculate true and predicted sums for comparison
            t = np.sum((targets[0] / scale).cpu().numpy().squeeze())  # Ground truth sum
            g = np.sum(outputs.cpu().numpy().squeeze())  # Predicted sum
            print(f'NO.{i}   true_sum={t}, get_sum={g}, abs={abs(t - g)}, rate={abs(1 - g / t)}')

            # Update metrics
            predict_num += g
            true_num += t
            rate += abs(1 - g / t)
            mae += abs(t - g)
            mse += abs(t - g) * abs(t - g)

    # Calculate average metrics across all batches
    mae /= len(val_loader)
    mse /= len(val_loader)
    predict_num /= len(val_loader)
    true_num /= len(val_loader)
    rate /= len(val_loader)

    # Log the results
    logging.info(
        f'test ---- Score: {math.exp(-rate):.3f}, MSE: {mse:.4f}, MAE: {mae:.4f}, Chicken_avg: {predict_num:.4f}')
    return math.exp(-rate)

In [12]:
model.load_state_dict(torch.load("/kaggle/working/best_model.pth", map_location=DEVICE))
model.to(DEVICE)
evaluate(model, val_loader, DEVICE, 100)

NO.0   true_sum=25.80451774597168, get_sum=23.648212432861328, abs=2.1563053131103516, rate=0.08356308937072754
NO.1   true_sum=18.838186264038086, get_sum=17.81068992614746, abs=1.027496337890625, rate=0.054543256759643555
NO.2   true_sum=19.0, get_sum=18.788433074951172, abs=0.21156692504882812, rate=0.011135101318359375
NO.3   true_sum=18.773645401000977, get_sum=16.60940170288086, abs=2.164243698120117, rate=0.11528092622756958
NO.4   true_sum=34.61363983154297, get_sum=34.60292434692383, abs=0.010715484619140625, rate=0.0003095865249633789
NO.5   true_sum=16.0, get_sum=15.259291648864746, abs=0.7407083511352539, rate=0.04629427194595337
NO.6   true_sum=19.820634841918945, get_sum=19.4328670501709, abs=0.3877677917480469, rate=0.01956385374069214
NO.7   true_sum=22.812883377075195, get_sum=22.461387634277344, abs=0.35149574279785156, rate=0.015407800674438477
NO.8   true_sum=19.999853134155273, get_sum=18.238201141357422, abs=1.7616519927978516, rate=0.08808326721191406
NO.9   true

2026-03-11 18:02:41,703 - INFO - test ---- Score: 0.951, MSE: 2.8987, MAE: 1.3716, Chicken_avg: 29.4296


NO.99   true_sum=26.9598445892334, get_sum=25.845407485961914, abs=1.1144371032714844, rate=0.04133695363998413


0.9512301537750563

In [4]:
from datasets import load_dataset

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test", 
                            data_dir="valandtest",
                            split="validation")

def collate_fn(batch): # The test datasets will not provide target densities
    return torch.stack([image_transform(item["image"]) for item in batch])

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / 100
        predictions.append(outputs.cpu().numpy())

pred_a = np.concatenate(predictions, axis=0)

del test_dataset
del test_loader
del predictions

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test", 
                            data_dir="valandtest",
                            split="test")  
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / 100
        predictions.append(outputs.cpu().numpy())

pred_b = np.concatenate(predictions, axis=0)

np.savez('submission.npz', pred_a=pred_a, pred_b=pred_b)

README.md:   0%|          | 0.00/446 [00:00<?, ?B/s]

valandtest/validation-00000-of-00001.par(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

valandtest/test-00000-of-00001.parquet:   0%|          | 0.00/168M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

NameError: name 'DataLoader' is not defined

In [42]:
%run /kaggle/input/datasets/alengevorgyan/aaaaaaaaaaaaaa/metrics.py

2026-03-18 17:24:36 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-18 17:24:37 - INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/c67fb0ac0cada3d502ef0f1f8779dc668897dcfb/README.md "HTTP/1.1 200 OK"
2026-03-18 17:24:37 - INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/c67fb0ac0cada3d502ef0f1f8779dc668897dcfb/README.md "HTTP/1.1 200 OK"


README.md:   0%|          | 0.00/480 [00:00<?, ?B/s]

2026-03-18 17:24:37 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/resolve/c67fb0ac0cada3d502ef0f1f8779dc668897dcfb/Task2_Chicken_Counting_LABEL.py "HTTP/1.1 404 Not Found"
2026-03-18 17:24:38 - INFO: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/ioaihsc/Task2_Chicken_Counting_LABEL.py "HTTP/1.1 404 Not Found"
2026-03-18 17:24:38 - INFO: HTTP Request: GET https://huggingface.co/api/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/revision/c67fb0ac0cada3d502ef0f1f8779dc668897dcfb "HTTP/1.1 200 OK"
2026-03-18 17:24:38 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/resolve/c67fb0ac0cada3d502ef0f1f8779dc668897dcfb/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-03-18 17:24:38 - INFO: HTTP Request: GET https://huggingface.co/api/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/tree/c67fb0ac0cada3d502ef0f1f8779dc668897dcfb/va

valandtest/validation-00000-of-00001.par(…):   0%|          | 0.00/9.28M [00:00<?, ?B/s]

2026-03-18 17:24:41 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/resolve/c67fb0ac0cada3d502ef0f1f8779dc668897dcfb/valandtest/test-00000-of-00001.parquet "HTTP/1.1 302 Found"


valandtest/test-00000-of-00001.parquet:   0%|          | 0.00/9.12M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

2026-03-18 17:24:43 - INFO: Evaluation ---- Score: 0.8953, MAE: 3.8494, Chicken_avg: 29.0193
2026-03-18 17:24:43 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-18 17:24:43 - INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/c67fb0ac0cada3d502ef0f1f8779dc668897dcfb/README.md "HTTP/1.1 200 OK"
2026-03-18 17:24:43 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/resolve/c67fb0ac0cada3d502ef0f1f8779dc668897dcfb/Task2_Chicken_Counting_LABEL.py "HTTP/1.1 404 Not Found"
2026-03-18 17:24:44 - INFO: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ioaihsc/Task2_Chicken_Counting_LABEL/ioaihsc/Task2_Chicken_Counting_LABEL.py "HTTP/1.1 404 Not Found"
2026-03-18 17:24:44 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_

Final Result: {'status': True, 'score': {'public_a': 0.8953363576406602, 'private_b': 0.9016585953715079}, 'msg': 'Success!'}


# Same With Attention

In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BottleneckAttention(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.mha = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, C, H, W]
        b, c, h, w = x.shape
        # Flatten to [B, H*W, C]
        x_flat = x.view(b, c, -1).permute(0, 2, 1)
        x_norm = self.norm(x_flat)
        
        # Self-Attention
        attn_out, _ = self.mha(x_norm, x_norm, x_norm)
        x_flat = x_flat + attn_out # Residual connection
        
        # Reshape back to [B, C, H, W]
        return x_flat.permute(0, 2, 1).view(b, c, h, w)

class AttentionUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        # Use your existing feature extractor (ResNet)
        from torchvision import models
        resnet = models.resnet34(pretrained=False)
        self.encoder = nn.ModuleList([
            nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu), # x1
            resnet.layer1, # x2
            resnet.layer2, # x3
            resnet.layer3  # x4 (bottleneck input)
        ])
        
        # ATTENTION BOTTLENECK (at the deepest layer)
        self.bottleneck_attn = BottleneckAttention(dim=256) 

        # Decoder layers (matches your notebook structure)
        self.up1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv1 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.final_conv = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        features = []
        for layer in self.encoder:
            x = layer(x)
            features.append(x)
        
        # Apply Attention to the bottleneck
        x = self.bottleneck_attn(features[-1])
        
        # Decoder with Skip Connections (simplified for brevity)
        d1 = self.up1(x)
        d1 = torch.cat([d1, features[-2]], dim=1) # Skip from x3
        d1 = F.relu(self.conv1(d1))
        
        d2 = self.up2(d1)
        d2 = torch.cat([d2, features[-3]], dim=1) # Skip from x2
        d2 = F.relu(self.conv2(d2))
        
        out = self.final_conv(d2)
        if out.shape[2:] != (180, 320):
            out = F.interpolate(out, size=(180, 320), mode='bilinear', align_corners=False)
        return F.relu(out)

In [37]:
def train_model(model, train_loader, val_loader, num_epochs=20):
    # 1. Setup for Memory Efficiency
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
    criterion = nn.MSELoss(reduction='sum')
    model.to(DEVICE)
    
    # Mixed Precision Scaler
    scaler = torch.cuda.amp.GradScaler()
    
    # Gradient Accumulation: if actual batch_size is 2, 
    # accumulation_steps=4 makes an effective batch size of 8.
    accumulation_steps = 4 
    
    best_mae = float('inf')
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        optimizer.zero_grad()
        
        for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}")):
            inputs = batch['image'].to(DEVICE)
            targets = batch['density'].to(DEVICE)
            
            # 2. Use Autocast for Mixed Precision
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                # Scale loss to account for accumulation
                loss = criterion(outputs, targets) / accumulation_steps
            
            # 3. Backward pass with Scaler
            scaler.scale(loss).backward()
            
            # 4. Step optimizer only after accumulation steps
            if (i + 1) % accumulation_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            train_loss += loss.item() * accumulation_steps
            
        # Validation Logic (remains similar but wrapped in no_grad)
        model.eval()
        val_mae = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs, targets = batch['image'].to(DEVICE), batch['density'].to(DEVICE)
                with torch.cuda.amp.autocast():
                    outputs = model(inputs)
                
                # Rescale prediction if you used a scale factor in collate_fn
                pred_count = outputs.sum().item()
                true_count = targets.sum().item()
                val_mae += abs(pred_count - true_count)
        
        avg_mae = val_mae / len(val_loader.dataset)
        logging.info(f"Epoch {epoch+1} - Train Loss: {train_loss/len(train_loader):.4f}, Val MAE: {avg_mae:.4f}")
        
        if avg_mae < best_mae:
            best_mae = avg_mae
            torch.save(model.state_dict(), "best_model.pth")
            
    return model

def predict_set(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            inputs = batch['image'].to(DEVICE)
            outputs = model(inputs)
            all_preds.append(outputs.cpu().numpy())
    return np.concatenate(all_preds, axis=0)

In [38]:
from datasets import load_dataset

train_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Train2", 
                            data_dir="train",
                            split="train")  

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def collate_fn(batch, scale=100):
    return {
        "image": torch.stack([image_transform(item["image"]) for item in batch]),
        "density": torch.stack([torch.tensor(item["density"], dtype=DTYPE).unsqueeze(0) * scale for item in batch])
    }

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

model = AttentionUNet()

2026-03-18 17:20:40 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_Train2/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-18 17:20:40 - INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ioaihsc/Task2_Chicken_Counting_Train2/377f01f034683afc5a49468001e75360af722393/README.md "HTTP/1.1 200 OK"
2026-03-18 17:20:40 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_Train2/resolve/377f01f034683afc5a49468001e75360af722393/Task2_Chicken_Counting_Train2.py "HTTP/1.1 404 Not Found"
2026-03-18 17:20:41 - INFO: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ioaihsc/Task2_Chicken_Counting_Train2/ioaihsc/Task2_Chicken_Counting_Train2.py "HTTP/1.1 404 Not Found"
2026-03-18 17:20:41 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_Train2/resolve/377f01f034683afc5a49468001e75360af722393/.huggingface.y

In [39]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [40]:
import torch.optim as optim

################################################################################
# Experiment Settings
################################################################################
learning_rate = 1e-4
lr_decay = 5e-6
weight_decay = 0.0001
save_path = "model.pth"

epochs = 20

model = AttentionUNet().to(DEVICE)
print('load model success')

optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=1 - lr_decay)

logging.info('Begin training single view model...')
train_model(model, train_loader, val_loader)
logging.info('Finished training single view model.')

2026-03-18 17:20:43 - INFO: Begin training single view model...


load model success


/tmp/ipykernel_55/1827668968.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1:   0%|          | 0/50 [00:00<?, ?it/s]

/tmp/ipykernel_55/1827668968.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_55/1827668968.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
2026-03-18 17:20:53 - INFO: Epoch 1 - Train Loss: 2265.6049, Val MAE: 839.2865


...

2026-03-18 17:23:46 - INFO: Epoch 18 - Train Loss: 105.3297, Val MAE: 171.3824


Epoch 19:   0%|          | 0/50 [00:00<?, ?it/s]

2026-03-18 17:23:56 - INFO: Epoch 19 - Train Loss: 93.8485, Val MAE: 125.8622


Epoch 20:   0%|          | 0/50 [00:00<?, ?it/s]

2026-03-18 17:24:06 - INFO: Epoch 20 - Train Loss: 88.5526, Val MAE: 128.1218
2026-03-18 17:24:06 - INFO: Finished training single view model.


In [41]:
from datasets import load_dataset

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test", 
                            data_dir="valandtest",
                            split="validation")

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def collate_fn(batch):
    images = torch.stack([test_transform(item["image"]) for item in batch])
    return images
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / 100
        predictions.append(outputs.cpu().numpy())

pred_a = np.concatenate(predictions, axis=0)

del test_dataset
del test_loader
del predictions

test_dataset = load_dataset("ioaihsc/Task2_Chicken_Counting_Test", 
                            data_dir="valandtest",
                            split="test")  
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

predictions = []
with torch.no_grad():
    for batch in tqdm(test_loader):
        outputs = model(batch.to(DEVICE)) / 100
        predictions.append(outputs.cpu().numpy())

pred_b = np.concatenate(predictions, axis=0)

np.savez('submission.npz', pred_a=pred_a, pred_b=pred_b)

2026-03-18 17:24:17 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_Test/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-18 17:24:17 - INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ioaihsc/Task2_Chicken_Counting_Test/decfcd957868ace7595df19638b04b0fe9deafbe/README.md "HTTP/1.1 200 OK"
2026-03-18 17:24:17 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_Test/resolve/decfcd957868ace7595df19638b04b0fe9deafbe/Task2_Chicken_Counting_Test.py "HTTP/1.1 404 Not Found"
2026-03-18 17:24:18 - INFO: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ioaihsc/Task2_Chicken_Counting_Test/ioaihsc/Task2_Chicken_Counting_Test.py "HTTP/1.1 404 Not Found"
2026-03-18 17:24:18 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_Test/resolve/decfcd957868ace7595df19638b04b0fe9deafbe/.huggingface.yaml "HTTP/1.1 

  0%|          | 0/7 [00:00<?, ?it/s]

2026-03-18 17:24:22 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_Test/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-18 17:24:22 - INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ioaihsc/Task2_Chicken_Counting_Test/decfcd957868ace7595df19638b04b0fe9deafbe/README.md "HTTP/1.1 200 OK"
2026-03-18 17:24:23 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_Test/resolve/decfcd957868ace7595df19638b04b0fe9deafbe/Task2_Chicken_Counting_Test.py "HTTP/1.1 404 Not Found"
2026-03-18 17:24:23 - INFO: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ioaihsc/Task2_Chicken_Counting_Test/ioaihsc/Task2_Chicken_Counting_Test.py "HTTP/1.1 404 Not Found"
2026-03-18 17:24:24 - INFO: HTTP Request: HEAD https://huggingface.co/datasets/ioaihsc/Task2_Chicken_Counting_Test/resolve/decfcd957868ace7595df19638b04b0fe9deafbe/.huggingface.yaml "HTTP/1.1 

  0%|          | 0/7 [00:00<?, ?it/s]